# ⚡ QualityBench — Colab T4 Benchmark Notebook

**Runtime: GPU → T4** (Runtime → Change runtime type → T4)

Run cells top-to-bottom. Each section corresponds to a day in the 7-day plan.

| Day | Section |
|-----|---------|
| 1 | Setup + HF FP16 baseline |
| 2 | INT8 + INT4 quantization |
| 3 | vLLM backend |
| 4 | llama.cpp + all evals |
| 5 | Pareto analysis |


---
## 0. Verify GPU

In [ ]:
!nvidia-smi
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

---
## 1. Clone repo + install dependencies (Day 1)

In [ ]:
# Clone (replace with your actual GitHub URL after pushing)
!git clone https://github.com/YOUR_USERNAME/qualitybench.git
%cd qualitybench

In [ ]:
# Core deps (torch already installed in Colab — skip to avoid CUDA version conflict)
!pip install -q \
    transformers==4.40.2 \
    accelerate==0.29.3 \
    bitsandbytes==0.43.1 \
    pyyaml==6.0.1 \
    mlflow==2.12.2 \
    plotly==5.21.0 \
    pandas==2.2.2 \
    tqdm==4.66.2 \
    nvidia-ml-py3==7.352.0

In [ ]:
# Quick smoke test
from benchmarks.profiler import Profiler, ProfileResult
from evals.eval_utils import EvalResult
from analysis.pareto import build_combined_df, is_pareto_optimal
print("✅ All imports OK")

### 1b. HF Transformers — FP16 Baseline

In [ ]:
!python benchmarks/run_hf_transformers.py \
    --config configs/benchmark_config.yaml \
    --quantization fp16

In [ ]:
import json
with open('results/hf_transformers_fp16.json') as f:
    r = json.load(f)
print(f"p50: {r['p50_latency_ms']:.1f}ms | p99: {r['p99_latency_ms']:.1f}ms | "
      f"{r['throughput_tokens_per_sec']:.1f} tok/s | {r['peak_gpu_memory_mb']:.0f} MB")

---
## 2. Quantization — INT8 + INT4 (Day 2)

In [ ]:
!python benchmarks/run_hf_transformers.py \
    --config configs/benchmark_config.yaml \
    --quantization int8

In [ ]:
!python benchmarks/run_hf_transformers.py \
    --config configs/benchmark_config.yaml \
    --quantization int4

In [ ]:
# Quick comparison table
import pandas as pd, json, glob

rows = []
for fpath in sorted(glob.glob('results/hf_transformers_*.json')):
    with open(fpath) as f:
        r = json.load(f)
    rows.append({
        'config': f"{r['backend']} / {r['quantization']}",
        'p50 (ms)': round(r['p50_latency_ms'], 1),
        'p99 (ms)': round(r['p99_latency_ms'], 1),
        'tok/s': round(r['throughput_tokens_per_sec'], 1),
        'GPU mem (MB)': round(r['peak_gpu_memory_mb'], 0),
    })
df = pd.DataFrame(rows)
display(df)

### 2b. First MMLU signal (100 samples)

In [ ]:
!pip install -q lm-eval==0.4.3 datasets==2.19.1

In [ ]:
!python evals/run_mmlu.py \
    --config configs/benchmark_config.yaml \
    --backend hf --quantization fp16

In [ ]:
!python evals/run_mmlu.py \
    --config configs/benchmark_config.yaml \
    --backend hf --quantization int8

---
## 3. vLLM Backend (Day 3)

> vLLM requires a specific CUDA build. Install separately to avoid conflicts.

In [ ]:
!pip install -q vllm==0.4.2

In [ ]:
!python benchmarks/run_vllm.py \
    --config configs/benchmark_config.yaml

In [ ]:
# Test batching — this is where vLLM's continuous batching advantage shows
!python benchmarks/run_vllm.py \
    --config configs/benchmark_config.yaml \
    --batch-size 4

In [ ]:
# Head-to-head: HF fp16 vs vLLM fp16
import json

with open('results/hf_transformers_fp16.json') as f:
    hf = json.load(f)
with open('results/vllm_fp16.json') as f:
    vl = json.load(f)

print("=== HF FP16 vs vLLM FP16 ===")
print(f"{'Metric':<25} {'HF FP16':>12} {'vLLM FP16':>12}")
print("-" * 50)
print(f"{'p50 latency (ms)':<25} {hf['p50_latency_ms']:>12.1f} {vl['p50_latency_ms']:>12.1f}")
print(f"{'p99 latency (ms)':<25} {hf['p99_latency_ms']:>12.1f} {vl['p99_latency_ms']:>12.1f}")
print(f"{'Throughput (tok/s)':<25} {hf['throughput_tokens_per_sec']:>12.1f} {vl['throughput_tokens_per_sec']:>12.1f}")
print(f"{'Peak GPU mem (MB)':<25} {hf['peak_gpu_memory_mb']:>12.0f} {vl['peak_gpu_memory_mb']:>12.0f}")
speedup = vl['throughput_tokens_per_sec'] / hf['throughput_tokens_per_sec']
print(f"\nvLLM throughput speedup: {speedup:.2f}x")

---
## 4. llama.cpp + All Evals (Day 4)

### 4a. Install llama-cpp-python with CUDA support

In [ ]:
!CMAKE_ARGS="-DLLAMA_CUDA=on" pip install llama-cpp-python==0.2.77 \
    --force-reinstall --no-cache-dir -q

### 4b. Download GGUF models

We use TheBloke's quantized Mistral-7B GGUF files.

In [ ]:
!pip install -q huggingface_hub
from huggingface_hub import hf_hub_download
import os

os.makedirs('models', exist_ok=True)

# Q4_K_M (~4.1 GB) — fastest, lowest quality
hf_hub_download(
    repo_id='TheBloke/Mistral-7B-v0.1-GGUF',
    filename='mistral-7b-v0.1.Q4_K_M.gguf',
    local_dir='./models'
)

# Q8_0 (~7.7 GB) — highest quality GGUF, the interesting comparison
hf_hub_download(
    repo_id='TheBloke/Mistral-7B-v0.1-GGUF',
    filename='mistral-7b-v0.1.Q8_0.gguf',
    local_dir='./models'
)

print('✅ GGUF models downloaded')

In [ ]:
!python benchmarks/run_llamacpp.py \
    --config configs/benchmark_config.yaml \
    --model-path models/mistral-7b-v0.1.Q4_K_M.gguf \
    --quantization q4_k_m

In [ ]:
!python benchmarks/run_llamacpp.py \
    --config configs/benchmark_config.yaml \
    --model-path models/mistral-7b-v0.1.Q8_0.gguf \
    --quantization q8_0

In [ ]:
# THE KEY COMPARISON: llama.cpp Q8_0 vs vLLM FP16 single-request latency
import json

with open('results/vllm_fp16.json') as f:
    vl = json.load(f)
with open('results/llamacpp_q8_0.json') as f:
    lc = json.load(f)

print("=== llama.cpp Q8_0 vs vLLM FP16 (single request) ===")
print(f"{'Metric':<25} {'vLLM FP16':>14} {'llama.cpp Q8_0':>14}")
print("-" * 54)
print(f"{'p50 latency (ms)':<25} {vl['p50_latency_ms']:>14.1f} {lc['p50_latency_ms']:>14.1f}")
print(f"{'p99 latency (ms)':<25} {vl['p99_latency_ms']:>14.1f} {lc['p99_latency_ms']:>14.1f}")
print(f"{'Throughput (tok/s)':<25} {vl['throughput_tokens_per_sec']:>14.1f} {lc['throughput_tokens_per_sec']:>14.1f}")
print(f"{'Peak GPU mem (MB)':<25} {vl['peak_gpu_memory_mb']:>14.0f} {lc['peak_gpu_memory_mb']:>14.0f}")

if lc['p50_latency_ms'] < vl['p50_latency_ms']:
    diff = (vl['p50_latency_ms'] - lc['p50_latency_ms']) / vl['p50_latency_ms'] * 100
    print(f"\n⚡ FINDING: llama.cpp Q8_0 is {diff:.1f}% faster on p50 latency than vLLM FP16!")
    print("   WHY: GGUF memory layout + fused dequantization; vLLM PagedAttention overhead")
    print("        dominates at batch_size=1. vLLM wins when concurrency > 1.")
else:
    print("\nvLLM FP16 is faster on this hardware — check n_gpu_layers setting for llama.cpp")

### 4c. Run all quality evals

In [ ]:
# MMLU on all HF configs
for quant in ['fp16', 'int8', 'int4']:
    print(f"\n--- MMLU: hf/{quant} ---")
    !python evals/run_mmlu.py \
        --config configs/benchmark_config.yaml \
        --backend hf --quantization {quant}

In [ ]:
# TruthfulQA on FP16 baseline
!python evals/run_truthfulqa.py \
    --config configs/benchmark_config.yaml \
    --backend hf --quantization fp16

---
## 5. Pareto Analysis (Day 5)

In [ ]:
!python analysis/pareto.py --results-dir results/

In [ ]:
# Display Pareto plot inline
import sys
sys.path.insert(0, '.')
from analysis.pareto import build_combined_df, plot_throughput_vs_memory, plot_quality_vs_efficiency, plot_latency_comparison

df = build_combined_df('results/')
print(df[['label','throughput_tokens_per_sec','p50_latency_ms','p99_latency_ms',
           'peak_gpu_memory_mb','efficiency_tps_per_gb']].to_string(index=False))

In [ ]:
fig = plot_throughput_vs_memory(df)
fig.show()

In [ ]:
fig2 = plot_quality_vs_efficiency(df)
if fig2:
    fig2.show()

---
## Save results to Google Drive (optional backup)

In [ ]:
# Uncomment to mount Drive and back up results
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r results/ /content/drive/MyDrive/qualitybench_results/
# print('✅ Results backed up to Drive')

---
## Commit results to GitHub

In [ ]:
import datetime
date = datetime.date.today().isoformat()
!git add results/*.json
!git commit -m f"data: add benchmark results {date}"
!git push origin main